# Notebook 11: Comparación Final de Representaciones
## BoW (TF-IDF) vs Word2Vec vs BERT

**Proyecto:** Análisis Morfosintáctico de Letras Musicales  
**Referencia metodológica:** Lab BoW – Word2Vec – BETO (Osvaldo González Chaves)

---

### Objetivo
Comparar las tres representaciones vectoriales sobre el **mismo corpus**,  
usando los embeddings ya calculados en los notebooks anteriores:

| Representación | Origen | Tipo |
|---|---|---|
| TF-IDF (BoW) | Calculado aquí desde `Lyrics` | Dispersa, sin semántica |
| Word2Vec | Cargado desde MongoDB (`embeddings.word2vec_avg`) | Densa, estática |
| BERT [CLS] | Cargado desde MongoDB (`embeddings.beto_cls`) | Densa, contextual |

**Tareas de evaluación:**
1. Clasificación de género (Regresión Logística)
2. Clustering K-Means + Silhouette Score
3. Visualización t-SNE comparativa
4. Reporte detallado por género

> *"La representación ideal es aquella que hace el problema más simple."*  
> — Presentación Semana 10


---
## 1. Imports y configuración

In [ ]:
import warnings, os, re
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
import sys
sys.path.append(os.path.abspath('../../../..'))
from src.data.mongo_storage import _get_default_collection

FIGS = Path('../../../..') / 'data' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2',
           '#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']

print('✓ Imports OK')


---
## 2. Cargar canciones con embeddings desde MongoDB

Solo se usan canciones que tienen **ambos** embeddings guardados  
(Word2Vec del notebook 09 y BERT del notebook 10).


In [ ]:
col = _get_default_collection()

docs = list(col.find(
    {
        'embeddings.word2vec_avg': {'$exists': True, '$ne': [], '$not': {'$size': 0}},
        'embeddings.beto_cls':     {'$exists': True, '$ne': [], '$not': {'$size': 0}},
        'Lyrics': {'$ne': None},
        'Genre':  {'$ne': None},
    },
    {'_id': 1, 'Song': 1, 'Artist': 1, 'Genre': 1,
     'Song year': 1, 'Lyrics': 1, 'embeddings': 1}
))

df = pd.DataFrame(docs)
df['Lyrics'] = df['Lyrics'].astype(str).str.strip()

# Extraer embeddings a columnas separadas
df['w2v_emb']  = df['embeddings'].apply(lambda x: x.get('word2vec_avg', []))
df['bert_emb'] = df['embeddings'].apply(lambda x: x.get('beto_cls', []))

# Filtrar filas con embeddings válidos
df = df[df['w2v_emb'].apply(len) > 0].copy()
df = df[df['bert_emb'].apply(len) > 0].copy()
df = df[df['Lyrics'].str.len() > 50].copy()
df = df.reset_index(drop=True)

# Solo géneros con >= 20 canciones (coherente con notebooks 09/10)
conteo = df['Genre'].value_counts()
generos_validos = conteo[conteo >= 20].index.tolist()
df = df[df['Genre'].isin(generos_validos)].reset_index(drop=True)

print(f'✓ {len(df):,} canciones con embeddings completos | {df["Genre"].nunique()} géneros')
print(df['Genre'].value_counts().to_string())
print()

# Verificar dimensiones de embeddings
w2v_dim  = len(df['w2v_emb'].iloc[0])
bert_dim = len(df['bert_emb'].iloc[0])
print(f'Dimensión Word2Vec : {w2v_dim}')
print(f'Dimensión BERT     : {bert_dim}')


---
## 3. Construir matrices de representación

Preparamos las tres matrices que compararemos:
- **TF-IDF:** generado ahora desde las letras crudas
- **Word2Vec:** cargado desde MongoDB (calculado en notebook 09)
- **BERT [CLS]:** cargado desde MongoDB (calculado en notebook 10)

> **Nota sobre TF-IDF:** usamos el mismo preprocesamiento de stopwords  
> que el lab de referencia (`stop_words='english'`) para ser comparables.


In [ ]:
# ── 1. TF-IDF (BoW) ──────────────────────────────────────────────────
print('Construyendo TF-IDF...')
tfidf_vec = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=2,
    sublinear_tf=True,   # log(1+tf) — mejora rendimiento en textos largos
)
X_tfidf = tfidf_vec.fit_transform(df['Lyrics'].tolist()).toarray()
print(f'  TF-IDF  : {X_tfidf.shape}  (dispersión: '
      f'{(X_tfidf == 0).mean()*100:.1f}%)')

# ── 2. Word2Vec (desde MongoDB) ───────────────────────────────────────
X_w2v = np.array(df['w2v_emb'].tolist())
print(f'  Word2Vec: {X_w2v.shape}')

# ── 3. BERT [CLS] (desde MongoDB) ────────────────────────────────────
X_bert = np.array(df['bert_emb'].tolist())
print(f'  BERT    : {X_bert.shape}')

# Etiquetas numéricas para sklearn
le     = LabelEncoder()
LABELS = le.fit_transform(df['Genre'].tolist())

REPRESENTACIONES = {
    'TF-IDF (BoW)': X_tfidf,
    'Word2Vec':     X_w2v,
    'BERT [CLS]':   X_bert,
}
print()
print('✓ Matrices listas')
print()
print('Resumen:')
print(f'  {"Representación":<20} {"Shape":<20} {"Tipo"}')
print('  ' + '-' * 55)
for nombre, X in REPRESENTACIONES.items():
    tipo = 'Dispersa' if nombre == 'TF-IDF (BoW)' else 'Densa'
    print(f'  {nombre:<20} {str(X.shape):<20} {tipo}')


---
## 4. Clasificación de género — Regresión Logística

Usamos **validación cruzada estratificada (5-fold)** en lugar de un split simple.  
Esto da una estimación más robusta del rendimiento real y evita depender  
de una partición particular de los datos.


In [ ]:
print('=== Clasificación de Género — Regresión Logística (CV 5-fold) ===')
print()

clf_scores  = {}   # accuracy media
clf_std     = {}   # desviación estándar
clf_reports = {}   # reporte detallado (en split 75/25 para el reporte por género)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nombre, X in REPRESENTACIONES.items():
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    # Validación cruzada 5-fold
    clf_cv = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    scores = cross_val_score(clf_cv, X_sc, LABELS, cv=cv, scoring='accuracy')
    clf_scores[nombre] = scores.mean()
    clf_std[nombre]    = scores.std()

    # Split 75/25 para obtener el classification_report detallado
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sc, LABELS, test_size=0.25, random_state=42, stratify=LABELS
    )
    clf_rep = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    clf_rep.fit(X_tr, y_tr)
    clf_reports[nombre] = classification_report(
        y_te, clf_rep.predict(X_te),
        target_names=le.classes_,
        output_dict=True
    )

    bar = '█' * int(scores.mean() * 30)
    print(f'  {nombre:<20s}: {scores.mean():.4f} ± {scores.std():.4f}  {bar}')

mejor_clf = max(clf_scores, key=clf_scores.get)
print()
print(f'🏆 Mejor: {mejor_clf}  ({clf_scores[mejor_clf]:.4f} ± {clf_std[mejor_clf]:.4f})')
print()
print('Interpretación:')
print('  • Accuracy alta no implica que la representación sea "mejor" conceptualmente')
print('  • TF-IDF suele ganar en clasificación por su alta dimensionalidad específica')
print('  • Word2Vec y BERT aportan semántica que TF-IDF no captura')


---
## 5. Clustering K-Means + Silhouette Score

El Silhouette Score mide qué tan bien separados están los clusters:
- **~1.0:** clusters muy compactos y bien separados
- **~0.0:** clusters solapados
- **< 0:** canciones asignadas al cluster incorrecto

> Con géneros musicales es normal obtener Silhouette Score bajo (<0.15)  
> porque los géneros comparten vocabulario y temáticas.


In [ ]:
n_clusters = len(generos_validos)
print(f'=== Clustering K-Means (k={n_clusters}) ===')
print()

sil_scores = {}

for nombre, X in REPRESENTACIONES.items():
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    # PCA antes de K-Means para reducir ruido en alta dimensión (especialmente TF-IDF)
    n_pca = min(100, X_sc.shape[1])
    pca   = PCA(n_components=n_pca, random_state=42)
    X_pca = pca.fit_transform(X_sc)

    km  = KMeans(n_clusters=n_clusters, random_state=42, n_init=15)
    lab = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, lab, sample_size=min(3000, len(X_pca)))
    sil_scores[nombre] = sil
    bar = '█' * max(0, int(sil * 100))
    print(f'  {nombre:<20s}: Silhouette = {sil:.4f}  {bar}')

mejor_sil = max(sil_scores, key=sil_scores.get)
print()
print(f'🏆 Mejor clustering: {mejor_sil}  (Silhouette = {sil_scores[mejor_sil]:.4f})')
print()
print('  Escala de referencia:')
print('  > 0.50  Estructura muy fuerte')
print('  > 0.25  Razonable')
print('  > 0.10  Débil (esperable en géneros musicales)')
print('  < 0.10  Sin estructura clara')


---
## 6. Gráfico comparativo — Clasificación y Clustering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
nombres = list(REPRESENTACIONES.keys())
colores = PALETTE[:len(nombres)]

# ── Panel izquierdo: Clasificación ───────────────────────────────────
accs  = [clf_scores[n] for n in nombres]
stds  = [clf_std[n]    for n in nombres]
bars  = axes[0].bar(nombres, accs, color=colores,
                    edgecolor='white', linewidth=1.5,
                    yerr=stds, capsize=6, error_kw={'linewidth': 2})
axes[0].set_ylim(0, min(1.0, max(accs) * 1.25))
axes[0].set_ylabel('Accuracy (CV 5-fold)', fontsize=12)
axes[0].set_title('Clasificación de Género\n(Logistic Regression, CV 5-fold)',
                   fontweight='bold')
for bar, acc, std in zip(bars, accs, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + std + 0.015,
                 f'{acc:.3f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')

# ── Panel derecho: Clustering ─────────────────────────────────────────
sils  = [sil_scores[n] for n in nombres]
bars2 = axes[1].bar(nombres, sils, color=colores,
                    edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title(f'Calidad de Clustering\n(K-Means, k={n_clusters})',
                   fontweight='bold')
for bar, sil in zip(bars2, sils):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{sil:.4f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')

plt.suptitle('Comparación de Representaciones: TF-IDF vs Word2Vec vs BERT',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'comparacion_representaciones.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. Visualización t-SNE — Las tres representaciones comparadas

Proyectamos cada matriz a 2D con t-SNE para ver visualmente  
qué tan bien cada representación **separa los géneros**.

Usamos PCA como paso previo para acelerar t-SNE y estabilizar el resultado.


In [ ]:
print('Calculando t-SNE para las 3 representaciones...')
print('(puede tardar varios minutos en CPU)\n')

# Muestra estratificada para que t-SNE sea manejable y balanceado
MUESTRA = 60  # canciones por género
df_plot = df.groupby('Genre', group_keys=False).apply(
    lambda x: x.sample(min(MUESTRA, len(x)), random_state=42)
).reset_index(drop=True)
LABELS_PLOT = le.transform(df_plot['Genre'].tolist())

def calcular_tsne(X_full: np.ndarray, idx_plot, nombre: str) -> np.ndarray:
    X_sub  = X_full[idx_plot]
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_sub)
    n_pca  = min(50, X_sc.shape[1])
    pca    = PCA(n_components=n_pca, random_state=42)
    X_pca  = pca.fit_transform(X_sc)
    print(f'  {nombre:<20}: PCA → {n_pca}d', end=' ... ', flush=True)
    tsne  = TSNE(n_components=2, random_state=42,
                 perplexity=min(30, len(X_pca)//4),
                 n_iter=1000)
    X_2d  = tsne.fit_transform(X_pca)
    print('OK')
    return X_2d

# Índices de la muestra en el dataframe completo
idx_plot = df_plot.index.tolist()
# Reconstruir índices posicionales en X matrices
pos_plot = [df.index.get_loc(i) for i in idx_plot]

tsne_res = {}
for nombre, X in REPRESENTACIONES.items():
    tsne_res[nombre] = calcular_tsne(X, pos_plot, nombre)

print()
print('Generando gráfico comparativo t-SNE...')

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
generos_plot = df_plot['Genre'].tolist()
generos_uniq = sorted(set(generos_plot))
color_dict   = dict(zip(generos_uniq, PALETTE[:len(generos_uniq)]))

for ax, (nombre, X_2d) in zip(axes, tsne_res.items()):
    for genero in generos_uniq:
        mask = [g == genero for g in generos_plot]
        xs = X_2d[mask, 0]
        ys = X_2d[mask, 1]
        ax.scatter(xs, ys, label=genero,
                   color=color_dict[genero],
                   alpha=0.65, s=22, zorder=3)
    ax.set_title(nombre, fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')

handles, labels_leg = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels_leg,
           loc='lower center', ncol=len(generos_uniq),
           fontsize=9, bbox_to_anchor=(0.5, -0.08))

plt.suptitle(f't-SNE: Separación entre géneros por representación\n'
             f'({MUESTRA} canciones por género, muestra estratificada)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'tsne_comparacion_representaciones.png',
            dpi=150, bbox_inches='tight')
plt.show()


---
## 8. Reporte detallado por género — mejor representación

Vemos qué géneros son más fáciles/difíciles de clasificar  
y si hay patrones de confusión entre géneros similares.


In [ ]:
print(f'=== Classification Report — {mejor_clf} ===')
print()

report = clf_reports[mejor_clf]
rows   = []
for genero in le.classes_:
    r = report.get(genero, {})
    rows.append({
        'Género':    genero,
        'Precision': round(r.get('precision', 0), 3),
        'Recall':    round(r.get('recall', 0), 3),
        'F1-Score':  round(r.get('f1-score', 0), 3),
        'Support':   int(r.get('support', 0)),
    })

df_report = pd.DataFrame(rows).sort_values('F1-Score', ascending=False)
print(df_report.to_string(index=False))
print()
print('Interpretación:')
print('  • F1 alto  → género fácil de identificar (vocabulario distintivo)')
print('  • F1 bajo  → género se confunde con otros (vocabulario compartido)')


In [ ]:
# Heatmap F1 por representación y género
f1_data = {}
for nombre, report in clf_reports.items():
    f1_data[nombre] = {g: round(report.get(g, {}).get('f1-score', 0), 3)
                       for g in le.classes_}

df_f1 = pd.DataFrame(f1_data).T  # representaciones × géneros

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(df_f1, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'F1-Score'})
ax.set_title('F1-Score por representación y género\n'
             '(verde = bien clasificado | rojo = difícil)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Representación')
ax.set_xlabel('Género')
plt.tight_layout()
plt.savefig(FIGS / 'f1_heatmap_representaciones.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 9. Dispersión TF-IDF — el problema fundamental de BoW

Aunque TF-IDF puede ganar en clasificación, tiene una limitación estructural:  
su espacio vectorial es **ortogonal** — palabras semánticamente cercanas  
tienen distancia idéntica a palabras no relacionadas.

Replicamos la demostración del lab de referencia adaptada al corpus musical.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Demostración de ortogonalidad en TF-IDF
print('=== El problema de BoW: ortogonalidad semántica ===')
print()

# Buscar en el corpus letras con estas palabras clave
def buscar_letra(keyword, df, n=1):
    mask = df['Lyrics'].str.lower().str.contains(keyword, regex=False)
    return df[mask].iloc[:n]['Lyrics'].tolist()

letras_amor    = buscar_letra(' love ', df)
letras_pasion  = buscar_letra(' passion ', df)
letras_dinero  = buscar_letra(' economics ', df)

if letras_amor and letras_pasion and letras_dinero:
    docs_demo = [letras_amor[0][:200], letras_pasion[0][:200], letras_dinero[0][:200]]
    etiquetas_demo = ['letra con "love"', 'letra con "passion"', 'letra con "economics"']

    tfidf_demo = TfidfVectorizer(stop_words='english').fit_transform(docs_demo)

    print('Similitud TF-IDF:')
    print(f'  "love" ↔ "passion"   : '
          f'{cos_sim(tfidf_demo[0], tfidf_demo[1])[0][0]:.4f}  '
          f'(semánticamente relacionadas, TF-IDF no lo sabe)')
    print(f'  "love" ↔ "economics" : '
          f'{cos_sim(tfidf_demo[0], tfidf_demo[2])[0][0]:.4f}  '
          f'(sin relación semántica)')
    print()
    print('→ TF-IDF trata "love" y "passion" igual que "love" y "economics"')
    print('  si no comparten tokens exactos.')
    print()

# Mostrar dispersión del espacio TF-IDF
print(f'Dispersión de X_tfidf : {(X_tfidf == 0).mean()*100:.1f}%')
print(f'Dispersión de X_w2v   : {(X_w2v == 0).mean()*100:.1f}%')
print(f'Dispersión de X_bert  : {(X_bert == 0).mean()*100:.1f}%')
print()
print('→ TF-IDF tiene miles de dimensiones casi siempre en cero.')
print('  Word2Vec y BERT usan vectores densos donde cada dimensión aporta.')


---
## 10. Resumen final

In [ ]:
print('=' * 65)
print('  RESUMEN — Comparación de Representaciones Vectoriales')
print('=' * 65)
print(f'  Corpus : {len(df):,} canciones | {len(generos_validos)} géneros')
print()

print('  CLASIFICACIÓN (Logistic Regression, CV 5-fold):')
for n, acc in sorted(clf_scores.items(), key=lambda x: -x[1]):
    star = '🏆' if n == mejor_clf else '  '
    print(f'  {star} {n:<22s} acc = {acc:.4f} ± {clf_std[n]:.4f}')

print()
print('  CLUSTERING (Silhouette Score, K-Means + PCA):')
for n, sil in sorted(sil_scores.items(), key=lambda x: -x[1]):
    star = '🏆' if n == mejor_sil else '  '
    print(f'  {star} {n:<22s} sil = {sil:.4f}')

print()
print('  CARACTERÍSTICAS FUNDAMENTALES:')
print()
print('  TF-IDF (BoW)')
print('  • Alta dimensionalidad (5000 features), muy dispersa')
print('  • No captura semántica: "love" ≠ "passion" si no comparten tokens')
print('  • Fuerte en clasificación por especificidad léxica por género')
print()
print('  Word2Vec')
print('  • Vectores densos (100d), estáticos por palabra')
print('  • Captura analogías y vecinos semánticos (fire→burn, dark→light)')
print('  • Una única representación por palabra, independiente del contexto')
print()
print('  BERT [CLS]')
print('  • Vectores densos (768d), contextuales por oración')
print('  • "fire" en "on fire tonight" ≠ "open fire on the enemy"')
print('  • Mejor para búsqueda semántica y polisemia')
print('  • Más costoso computacionalmente')
print()
print('  CONCLUSIÓN GENERAL:')
print('  No hay una representación universalmente mejor.')
print('  La elección depende de la tarea:')
print('  • Clasificación de género  → TF-IDF o BERT fine-tuned')
print('  • Búsqueda semántica       → BERT [CLS]')
print('  • Análisis de vocabulario  → Word2Vec por género')
print('  • Línea base rápida        → TF-IDF')
print('=' * 65)
